# Equilibrium conversion by reported by-products

Run from this folder with the qiao kernel. Read the parent `encoded_dataset.xlsx` and the local copy of the original AramcoMech 3.0 thermodynamic YAML. Evaluate TP equilibrium for one ideal-gas phase with a binary H₂/CO₂ inlet at the reported ratio. The input labels indicate some N₂- or Ar-diluted feeds, and the calculation uses the original binary-feed convention.

The equilibrium conversion is calculated from the fraction of gas-phase carbon remaining in CO₂, with two carbon atoms per DME or methyl formate molecule. ATE is the observed conversion divided by the calculated equilibrium conversion. The Cantera solver settings and the input hashes are written to the result workbook.

[Cantera ThermoPhase API](https://cantera.org/3.0/sphinx/html/cython/thermo.html) · [AramcoMech 3.0 thermodynamic data](https://universityofgalway.ie/combustionchemistrycentre/mechanismdownloads/)

The row-wise species set uses all mapped `Byproduct` entries. The sensitivity cases examine removal of CH₄ or DME from reported sets and addition of either or both to a uniform CO baseline. The cluster analysis compares scenarios on identical records and fits each scenario's scaling and K-means on the same training rows.

In [ ]:
from functools import lru_cache
from hashlib import sha256
from pathlib import Path
import cantera as ct
import numpy as np
import pandas as pd
from IPython.display import display

from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

import matplotlib as mpl
import matplotlib.pyplot as plt

In [ ]:


dataset_path = Path("derived") / "dataset.xlsx"  # Path to the required dataset workbook.
thermo_path = Path("thermo") / "AramcoMech3.0.yaml"  # Path to the Cantera mechanism and thermodynamic data.
output_dir = Path("equilibrium_results")  # Folder for generated results.
output_dir.mkdir(exist_ok=True)
point_id = "Data point number"  # Unique experimental-record identifier column.
condition_columns = ["T", "P", "H2/CO2"]  
source_columns = [  
    point_id, "Catalyst number", "Reference DOI", *condition_columns, "Byproduct",
    "Feed Composition", "Con", "Sel", "EqCon", "Con/EqCon",
]
base_species = ("CO2", "H2", "CH3OH", "H2O") 
elements = ("C", "H", "O")  
eq_tolerance = 1e-6  
ate_tolerance = 1e-7  
solver_rtol = 1e-9  

data = pd.read_excel(dataset_path, sheet_name="data", usecols=source_columns, engine="openpyxl")  # Input records for this analysis.
if data[point_id].isna().any() or not data[point_id].is_unique:
    raise ValueError("Data point numbers must be present and unique.")
numeric = [*condition_columns, "Con", "Sel", "EqCon", "Con/EqCon"]  
data[numeric] = data[numeric].apply(pd.to_numeric)
if not np.isfinite(data[numeric].to_numpy(dtype=float)).all():
    raise ValueError("Conditions, observed outcomes and reference equilibrium values must be finite.")
if data[[*condition_columns, "EqCon"]].le(0).any().any():
    raise ValueError("Temperature, pressure, feed ratio and stored equilibrium conversion must be positive.")
if not np.allclose(data["Con"] / data["EqCon"], data["Con/EqCon"], atol=ate_tolerance, rtol=0):
    raise ValueError("The stored ATE column does not agree with Con divided by the stored EqCon.")

species_db = {  
    item.name: item for item in ct.Species.list_from_yaml(thermo_path.read_text(encoding="utf-8-sig"), "species")  
}


@lru_cache(maxsize=None)
def gas_phase(names):  # Construct an ideal-gas phase for the allowed species.
    species = [species_db[name] for name in names]  # Chemical species admitted to the equilibrium calculation. Current item name.
    return ct.Solution(thermo="ideal-gas", species=species)


@lru_cache(maxsize=None)
def equilibrium(temp, pressure_mpa, h2_co2, extras):  # Return the constant-temperature, constant-pressure equilibrium result.
    names = base_species + extras  
    gas = gas_phase(names)  # Cantera ideal-gas mixture.
    if not gas.min_temp <= temp <= gas.max_temp:
        raise ValueError(f"Temperature {temp} K lies outside the thermodynamic range of {names}.")
    pressure_pa = pressure_mpa * 1e6  
    gas.TPX = temp, pressure_pa, {"CO2": 1, "H2": h2_co2}
    initial = np.array([gas.elemental_mass_fraction(element) for element in elements])  
    gas.equilibrate("TP", solver="auto", rtol=solver_rtol)
    final = np.array([gas.elemental_mass_fraction(element) for element in elements])  
    elemental_error = float(np.max(np.abs(final - initial)))  
    if elemental_error > 1e-8:
        raise ValueError(f"Elemental mass-fraction error {elemental_error:.3g} for {temp}, {pressure_mpa}, {h2_co2}.")
    if not np.isclose(gas.T, temp, rtol=1e-8) or not np.isclose(gas.P, pressure_pa, rtol=1e-8):
        raise ValueError("The solution departed from the prescribed temperature or pressure.")
    carbon_counts = np.array([gas.n_atoms(index, "C") for index in range(gas.n_species)])  # Carbon counts. Current position or index label.
    gas_carbon = float(gas.X @ carbon_counts)  
    conversion = 100 * (1 - gas["CO2"].X[0] / gas_carbon)  
    if not np.isfinite(conversion) or not 0 < conversion <= 100:
        raise ValueError(f"Invalid calculated equilibrium conversion {conversion}.")
    return conversion, elemental_error


def calculate_case(extras_by_row):  # Calculate equilibrium conversion for one record and species set.
    temperatures, pressures, ratios = (data[column] for column in condition_columns) 
    rows = []  # Rows accumulated for the result table.
    for point, temp, pressure, ratio, extras in zip(data[point_id], temperatures, pressures, ratios, extras_by_row, strict=True):  # Point; Temp; Pressure; Ratio; Extras.
        try:
            conversion, error = equilibrium(temp, pressure, ratio, extras)  
        except (ct.CanteraError, ValueError, KeyError) as exc:  
            raise RuntimeError(f"Equilibrium calculation failed for data point {point}, extras={extras}.") from exc
        rows.append((conversion, error, ", ".join(base_species + extras)))
    result = pd.DataFrame(rows, columns=["EqCon (%)", "Max element mass fraction error", "Gas species"], index=data.index)  
    result["ATE"] = data["Con"] / result["EqCon (%)"]
    result["EqCon difference (pp)"] = result["EqCon (%)"] - data["EqCon"]
    result["ATE difference"] = result["ATE"] - data["Con/EqCon"]
    result["EqCon reproduced"] = result["EqCon difference (pp)"].abs().le(eq_tolerance)
    result["ATE reproduced"] = result["ATE difference"].abs().le(ate_tolerance)
    return result


def case_summary(result):  # Summarize conversion reproduction and ATE for one equilibrium case.
    error = result["EqCon difference (pp)"].abs()  
    return {
        "Records": len(result),
        "EqCon reproduced": int(result["EqCon reproduced"].sum()),
        "ATE reproduced": int(result["ATE reproduced"].sum()),
        "EqCon MAE (pp)": error.mean(),
        "EqCon max absolute difference (pp)": error.max(),
        "Mean absolute ATE difference": result["ATE difference"].abs().mean(),
        "ATE > 1": int(result["ATE"].gt(1).sum()),
        "Max element mass fraction error": result["Max element mass fraction error"].max(),
    }


metadata = pd.DataFrame([ 
    ("Dataset", dataset_path.name),
    ("Dataset SHA-256", sha256(dataset_path.read_bytes()).hexdigest()),
    ("Thermodynamic data", thermo_path.name),
    ("Thermodynamic data SHA-256", sha256(thermo_path.read_bytes()).hexdigest()),
    ("Cantera version", ct.__version__),
    ("Phase", "Single ideal gas with the listed gas species"),
    ("Temperature and pressure", "T in K, P in MPa converted to Pa. Isothermal, isobaric equilibrium."),
    ("Feed assumption", "Binary H2/CO2 at the reported ratio. N2/Ar dilution is not quantified in the encoded input."),
    ("Solver", f"equilibrate('TP', solver='auto', rtol={solver_rtol:g})"),
    ("Conversion", "100 * [1 - x(CO2) / sum_i(n_C,i * x_i)] for this CO2-only carbon feed"),
    ("ATE", "Observed conversion (%) / equilibrium conversion (%)"),
    ("EqCon reproduction tolerance (pp)", eq_tolerance),
    ("ATE reproduction tolerance", ate_tolerance),
], columns=["Setting", "Value"])

In [ ]:
aliases = {  
    "CO": "CO", "CH4": "CH4", "DME": "CH3OCH3", "CH3OCH3": "CH3OCH3",
    "HCHO": "CH2O", "CH2O": "CH2O", "HCOOCH3": "CH3OCHO", "CH3OCHO": "CH3OCHO",
}
subscripts = str.maketrans("₀₁₂₃₄₅₆₇₈₉", "0123456789") 


def reported_species(value, point):  # Translate reported by-product names into mechanism species names.
    if pd.isna(value) or not str(value).strip():
        raise ValueError(f"Missing Byproduct entry at data point {point}.")
    tokens = {token.strip() for token in str(value).translate(subscripts).upper().split(",")}  
    unknown = tokens.difference(aliases)  #
    if unknown:
        raise ValueError(f"Unmapped Byproduct labels at data point {point}: {sorted(unknown)}")
    return tuple(sorted({aliases[token] for token in tokens}, key=lambda name: (name != "CO", name)))  


reported = [reported_species(value, point) for value, point in zip(data["Byproduct"], data[point_id], strict=True)]  
cases = {  # Cases.
    "CO": [("CO",)] * len(data),
    "Reported": reported,
    "Legacy CO+reported CH4": [tuple(name for name in row if name in {"CO", "CH4"}) for row in reported],  
    "Reported without CH4": [tuple(name for name in row if name != "CH4") for row in reported],  
    "Reported without DME": [tuple(name for name in row if name != "CH3OCH3") for row in reported],  
    "CO+CH4 all rows": [("CO", "CH4")] * len(data),
    "CO+DME all rows": [("CO", "CH3OCH3")] * len(data),
    "CO+CH4+DME all rows": [("CO", "CH3OCH3", "CH4")] * len(data),
}
calculated = {name: calculate_case(extras) for name, extras in cases.items()}  
reproduction = pd.DataFrame([{"Scenario": name, **case_summary(result)} for name, result in calculated.items()])  
species_sets = (  # Species sets.
    calculated["Reported"][["Gas species"]].join(data["Byproduct"])
    .value_counts().rename("Records").reset_index()
)
species_names = sorted(set(base_species).union(*(set(row) for row in reported)))  
species_data = pd.DataFrame([  
    {"Species": name, "Composition": str(species_db[name].composition),
     "NASA data minimum T (K)": species_db[name].thermo.min_temp,
     "NASA data maximum T (K)": species_db[name].thermo.max_temp,
     "Reference pressure (Pa)": species_db[name].thermo.reference_pressure}
    for name in species_names  
])
display(reproduction)

In [ ]:

train, test = train_test_split(data.index, test_size=0.2, random_state=0)  # Development records or their indices; Test.
cluster_ids = [0, 1, 2, 3]  
splits = pd.Series("Test", index=data.index, name="Split")  
splits.loc[train] = "Train"


def assign_clusters(ate):  # Fit and assign K-means regimes for one equilibrium scenario.
    values = pd.DataFrame({"ATE": ate, "Sel": data["Sel"]})  # Values used in the current calculation.
    scaler = StandardScaler().fit(values.loc[train])  # Predictor or outcome standardization fitted on training data.
    with threadpool_limits(limits=1):
        kmeans = KMeans(n_clusters=4, n_init=10, random_state=0).fit(scaler.transform(values.loc[train]))  
        labels = kmeans.predict(scaler.transform(values))  
    centers = kmeans.cluster_centers_  
    ranked = np.argsort(centers[:, 0])  
    labels_by_center = np.empty(4, dtype=int) 
    for side, ids in zip((ranked[:2], ranked[2:]), ((0, 3), (1, 2))):  
        high, low = side[np.argsort(centers[side, 1])[::-1]]  # Upper bound for the displayed values; Lower bound for the displayed values.
        labels_by_center[high], labels_by_center[low] = ids
    return pd.Series(labels_by_center[labels], index=data.index, name="Cluster")


ate_values = {"Stored": data["Con/EqCon"], **{name: table["ATE"] for name, table in calculated.items()}}  # ATE values. Current item name; Result table.
labels = {name: assign_clusters(ate) for name, ate in ate_values.items()}  # Group or display labels. Current item name; Measured conversion divided by equilibrium conversion.
cluster_rows = []  
for name, assigned in labels.items():  
    for cluster, index in assigned.groupby(assigned).groups.items():  
        ate = ate_values[name].loc[index]  
        cluster_rows.append({
            "Scenario": name, "Cluster": cluster, "N": len(index), "Mean ATE": ate.mean(),
            "Mean Sel (%)": data.loc[index, "Sel"].mean(), "ATE > 1": int(ate.gt(1).sum()),
        })
cluster_summary = pd.DataFrame(cluster_rows)  # Record counts and mean outcomes by regime.

full_sample = pd.Series(True, index=data.index)  # Full sample.
pairings = [("Stored", name, full_sample, "All records") for name in calculated]  # Pairings. Current item name.
pairings += [
    ("CO", "Reported", full_sample, "All records"),
    ("Reported", "Reported without CH4", pd.Series(["CH4" in row for row in reported]), "CH4 reported"),  
    ("Reported", "Reported without DME", pd.Series(["CH3OCH3" in row for row in reported]), "DME reported"), 
    ("CO", "CO+CH4 all rows", full_sample, "All records"),
    ("CO", "CO+DME all rows", full_sample, "All records"),
    ("CO+CH4 all rows", "CO+CH4+DME all rows", full_sample, "All records"),
    ("CO+DME all rows", "CO+CH4+DME all rows", full_sample, "All records"),
]
comparisons, transitions = [], []  
for before, after, mask, subset in pairings: 
    a, b = labels[before][mask], labels[after][mask]  
    if a.empty:
        continue
    x, y = ate_values[before][mask], ate_values[after][mask]  # Horizontal coordinate or predictor values; Observed response.
    tags = {"Reference": before, "Scenario": after, "Subset": subset}  
    comparisons.append({
        **tags, "N": len(a), "Mean absolute ATE change": (y - x).abs().mean(),
        "Max absolute ATE change": (y - x).abs().max(),
        "Reference ATE > 1": int(x.gt(1).sum()), "Scenario ATE > 1": int(y.gt(1).sum()),
        "Changed cluster": int(a.ne(b).sum()), "Changed cluster (%)": 100 * a.ne(b).mean(),
        "ARI": adjusted_rand_score(a, b), "Entered Cluster 1": int((a.ne(1) & b.eq(1)).sum()),
        "Left Cluster 1": int((a.eq(1) & b.ne(1)).sum()),
    })
    matrix = pd.crosstab(a.rename("Reference cluster"), b.rename("Scenario cluster")).reindex(  # Matrix.
        index=cluster_ids, columns=cluster_ids, fill_value=0,
    )
    for i in cluster_ids:  # Loop position.
        for j in cluster_ids:  # Second loop position.
            transitions.append({**tags, "Reference cluster": i, "Scenario cluster": j, "N": int(matrix.loc[i, j])})
comparisons = pd.DataFrame(comparisons)
transitions = pd.DataFrame(transitions)
display(comparisons)

In [ ]:
output = data.assign(Split=splits, **{"Stored cluster": labels["Stored"]}) 
for name, result in calculated.items(): 
    output[f"{name} | EqCon (%)"] = result["EqCon (%)"]
    output[f"{name} | ATE"] = result["ATE"]
    output[f"{name} | cluster"] = labels[name]
output["Reported gas species"] = calculated["Reported"]["Gas species"]
output["Reported EqCon difference (pp)"] = calculated["Reported"]["EqCon difference (pp)"]
output["Reported ATE difference"] = calculated["Reported"]["ATE difference"]
output["Reported EqCon reproduced"] = calculated["Reported"]["EqCon reproduced"]
output["Reported max element mass fraction error"] = calculated["Reported"]["Max element mass fraction error"]

label_summary = (  
    output.groupby("Byproduct", dropna=False)
    .agg(N=(point_id, "size"), Matching_EqCon=("Reported EqCon reproduced", "sum"),
         Mean_EqCon_change_pp=("Reported EqCon difference (pp)", "mean"),
         Mean_ATE_change=("Reported ATE difference", "mean"))
    .reset_index()
)
sensitivity_notes = pd.DataFrame([  
    ("Reported aliases", ", ".join(f"{label} → {name}" for label, name in aliases.items() if label != name)),  
    ("Legacy case", "CO plus CH4 when listed, corresponding to the original script for the observed Byproduct labels."),
    ("Omission cases", "Remove CH4 or DME from the reported species set. Refit each scenario on the same training records."),
    ("Uniform cases", "Add CH4 and/or DME as allowable species on all rows. Used for sensitivity to the admissible species set."),
    ("Clustering", "StandardScaler and four-cluster K-means, fit on the original 80% train split, seed=0, n_init=10."),
    ("Label assignment", "Order cluster centers into low/high ATE pairs, then by selectivity within each pair."),
    ("Cluster comparison", "ARI is label-invariant. Changed cluster counts use the relabeled cluster IDs."),
], columns=metadata.columns)
with pd.ExcelWriter(output_dir / "eqcon_byproduct.xlsx", engine="openpyxl") as writer:  # Excel writer for the result workbook.
    output.to_excel(writer, sheet_name="by_point", index=False)
    reproduction.to_excel(writer, sheet_name="reproduction", index=False)
    label_summary.to_excel(writer, sheet_name="byproduct_summary", index=False)
    species_sets.to_excel(writer, sheet_name="species_sets", index=False)
    species_data.to_excel(writer, sheet_name="species_thermo", index=False)
    cluster_summary.to_excel(writer, sheet_name="cluster_summary", index=False)
    comparisons.to_excel(writer, sheet_name="cluster_comparison", index=False)
    transitions.to_excel(writer, sheet_name="cluster_transitions", index=False)
    pd.concat([metadata, sensitivity_notes]).to_excel(writer, sheet_name="method", index=False)

In [ ]:

co = calculated["CO"]  # Co.
rowwise = calculated["Reported"]  
has_ch4 = pd.Series(["CH4" in row for row in reported])  
has_dme = pd.Series(["CH3OCH3" in row for row in reported])  
only_co = pd.Series([row == ("CO",) for row in reported])  
styles = [  # Styles.
    ("CO only", "#7f7f7f", only_co),
    ("CH$_4$ reported", "#1f77b4", has_ch4 & ~has_dme),
    ("DME reported", "#d94801", has_dme & ~has_ch4),
    ("CH$_4$ and DME reported", "#2ca02c", has_ch4 & has_dme),
    ("Other species reported", "#6a51a3", ~(only_co | has_ch4 | has_dme)),
]

with mpl.rc_context({"font.family": ["Arial", "DejaVu Sans"], "axes.labelweight": "bold",
                     "axes.titleweight": "bold", "axes.linewidth": 1.1}):
    fig, axes = plt.subplots(1, 3, figsize=(13, 4.4), layout="constrained")  # Matplotlib figure; Plot axes arranged by panel.
    for ax, column, description, letter in [  # Plot axes; Current column name; Description; Letter.
        (axes[0], "EqCon (%)", "equilibrium conversion (%)", "a"),
        (axes[1], "ATE", "ATE", "b"),
    ]:
        for label, color, mask in styles:  # Label; Color assigned to the current series.
            if not mask.any():
                continue
            ax.scatter(co.loc[mask, column], rowwise.loc[mask, column],
                       s=15, alpha=0.55, color=color, linewidths=0, label=label)
        lo = min(co[column].min(), rowwise[column].min())  
        hi = max(co[column].max(), rowwise[column].max()) 
        pad = 0.04 * (hi - lo)  
        bounds = (lo - pad, hi + pad)  
        ax.plot(bounds, bounds, linestyle="--", color="black", linewidth=0.9)
        if column == "ATE":
            ax.axhline(1, color="0.6", linestyle=":", linewidth=0.8)
            ax.axvline(1, color="0.6", linestyle=":", linewidth=0.8)
        ax.set(xlabel=f"CO-only {description}", ylabel=f"Byproduct {description}",
               xlim=bounds, ylim=bounds, title=f"({letter})")
        ax.set_aspect("equal", adjustable="box")
    axes[0].legend(loc="lower right", frameon=False, fontsize=8)

    matrix = pd.crosstab(labels["CO"], labels["Reported"]).reindex(index=cluster_ids, columns=cluster_ids, fill_value=0)
    ax = axes[2]
    ax.imshow(matrix, cmap="Blues", vmin=0)
    for i in cluster_ids:
        for j in cluster_ids:
            value = matrix.iat[i, j]  
            ax.text(j, i, str(value), ha="center", va="center",
                    color="white" if value > matrix.to_numpy().max() / 2 else "black")
    ax.set(xticks=cluster_ids, yticks=cluster_ids, xlabel="Byproduct cluster",
           ylabel="CO-only cluster", title="(c)")
    fig.savefig(output_dir / "EqCon_ATE_Sensitivity.tif", dpi=600, bbox_inches="tight",
                pil_kwargs={"compression": "tiff_lzw"})
    plt.show()
    plt.close(fig)